In [ ]:
!pip install plotly pandas requests nbformat -q

import requests
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta

API_URL = 'https://djtnqbvkhqftmtnsityx.supabase.co/rest/v1/'
API_KEY = 'sb_publishable_NnOjc1iJWmA7j418h79mEg_AHh9h49u'
HEADERS = {'apikey': API_KEY, 'Authorization': f'Bearer {API_KEY}'}

def query_supabase(endpoint, select='*', filters=None, limit=10000):
    """Query Supabase REST API with filters."""
    params = {}
    if select: params['select'] = select
    if filters:
        for k, v in filters.items():
            params[k] = f'eq.{v}'
    if limit: params['limit'] = limit
    r = requests.get(f'{API_URL}{endpoint}', headers=HEADERS, params=params)
    if r.status_code == 200:
        return pd.DataFrame(r.json())
    else:
        print(f'Error {r.status_code}: {r.text[:200]}')
        return pd.DataFrame()

In [ ]:
# Query venue comparison — cross-venue spreads
df = query_supabase(
    'venue_comparison',
    select='date,symbol,venue,avg_rate_bps,max_cross_spread_bps,arb_venue_pair',
    limit=50000
)

df['date'] = pd.to_datetime(df['date'])
df['avg_rate_bps'] = pd.to_numeric(df['avg_rate_bps'], errors='coerce')
df['max_cross_spread_bps'] = pd.to_numeric(df['max_cross_spread_bps'], errors='coerce')
df = df.dropna(subset=['max_cross_spread_bps'])
df_pos = df[df['max_cross_spread_bps'] > 0].copy()

print(f"Loaded {len(df_pos):,} rows with positive cross-venue spreads")
print(f"Date range: {df_pos['date'].min().date()} → {df_pos['date'].max().date()}")
df_pos.head()

In [ ]:
# Line chart: max cross-venue spread over time by symbol
spread_daily = df_pos.groupby(['date', 'symbol'])['max_cross_spread_bps'].max().reset_index()

fig = px.line(
    spread_daily, x='date', y='max_cross_spread_bps', color='symbol',
    title='Max Cross-Venue Spread Over Time',
    labels={'max_cross_spread_bps': 'Spread (bps)', 'date': 'Date'},
    template='plotly_dark'
)
fig.update_layout(height=500, hovermode='x unified')
fig.show()

In [ ]:
# Heatmap: correlation matrix of avg funding rates across venues
pivot = df.pivot_table(index='date', columns='venue', values='avg_rate_bps', aggfunc='mean')
corr = pivot.corr().round(3)

fig = px.imshow(
    corr, text_auto='.2f', color_scale='RdYlGn', aspect='auto',
    title='Funding Rate Correlation Across Venues',
    template='plotly_dark'
)
fig.update_layout(height=450)
fig.show()

In [ ]:
# Top 10 arbitrage opportunity days — highest cross-venue spreads
top10 = (
    df_pos.nlargest(10, 'max_cross_spread_bps')[
        ['date', 'symbol', 'venue', 'max_cross_spread_bps', 'arb_venue_pair']
    ]
    .reset_index(drop=True)
)
top10['date'] = top10['date'].dt.strftime('%Y-%m-%d')

print("Top 10 Arbitrage Opportunity Days (Highest Cross-Venue Spread)")
print("=" * 70)
top10